In [1]:
!pip install numpy scipy torch torchvision torchaudio pennylane scikit-learn tqdm --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 55.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 935.6/935.6 kB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 65.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 79.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 107.4 MB/s eta 0:00:0000:0100:01


In [2]:
import pennylane as qml
print(qml.version())

0.44.0


In [ ]:
"""
hybrid_q_tomography_3qubit.py
==============================
Geometric Latent Space Tomography — 3-Qubit Extension

Architecture corrections from 2-qubit version:
  [FIX 1] Decoder now inserts learnable DepolarizingChannel noise after each
           unitary layer, making the output a *genuinely mixed* quantum state.
           default.mixed is now legitimately required (density matrix sim needed
           for noise channels). diff_method='backprop' is used (adjoint is
           incompatible with noise channels).
  [FIX 2] evaluate_fidelity is fully GPU-batched via torch.linalg.eigh —
           the original per-sample scipy.linalg.sqrtm loop is eliminated.
  [FIX 3] metric_preservation_loss is fully vectorised (no Python pair loop).
  [FIX 4] Bures distance matrix for manifold analysis is GPU-batched.
  [NEW]   Shot noise robustness evaluation via analytical variance injection.
  [NEW]   Dataset generation is parallelised (joblib) and file-cached.
  [NEW]   eval_every=5 reduces evaluation overhead on Kaggle T4.
"""

# ============================================================================
# Imports
# ============================================================================
import os
import random
import time
import pickle

import numpy as np
from scipy.linalg import sqrtm
import scipy.linalg
from scipy.stats import pearsonr, spearmanr
from scipy.linalg import orthogonal_procrustes

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import pennylane as qml
from tqdm import tqdm

from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.manifold import MDS

import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from mpl_toolkits.mplot3d import Axes3D

try:
    from joblib import Parallel, delayed
    HAS_JOBLIB = True
except ImportError:
    HAS_JOBLIB = False
    print("Warning: joblib not found — dataset generation will be single-threaded.")

# ============================================================================
# Section 1 — Configuration
# ============================================================================

# --- Qubit system ---
n_qubits        = 3
dim             = 2 ** n_qubits          # 8
n_pauli_nonid   = 4 ** n_qubits - 1      # 63

# --- Circuit architecture ---
n_layers        = 6
n_angles        = n_qubits * 3 * n_layers   # 54  (RX, RY, RZ per qubit per layer)
n_noise_params  = n_layers                  # 6   (one depolarising strength per layer)
n_circuit_params = n_angles + n_noise_params # 60
NOISE_MAX       = 0.15   # Upper bound on per-layer depolarising probability

# --- Latent space ---
latent_dim      = 20

# --- Dataset (reduced for 3-qubit compute cost) ---
n_train         = 1500
n_val           = 400
batch_size      = 64
DATASET_CACHE   = '/kaggle/working/dataset_3q_cache.npz'
N_JOBS_DATA     = 4      # joblib workers for dataset generation

# --- Training ---
max_epochs              = 300
target_fidelity         = 0.93
early_stop_patience     = 16   # in evaluation steps (each step = eval_every epochs)
lr                      = 3e-3
weight_decay            = 1e-5
lambda_metric           = 0.06
use_uhlmann_loss        = True
eval_every              = 5    # evaluate fidelity every N epochs to save time

# --- Shot noise ---
SHOT_COUNTS     = [None, 10_000, 1_000, 100]
N_EVAL_SHOTS    = 80     # samples used for shot-noise evaluation

# --- Manifold analysis ---
n_vis_samples   = 120    # samples used for Bures distance matrix / MDS
n_geodesic_pairs = 400   # pairs for geodesic-Bures correlation

# --- Misc ---
shots           = None   # training uses exact (analytical) expectations
seed            = 42
device_torch    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTDIR          = '/kaggle/working'

print(f"System  : {n_qubits} qubits | dim={dim} | Pauli observables={n_pauli_nonid}")
print(f"Circuit : {n_layers} layers | {n_angles} angles + {n_noise_params} noise = {n_circuit_params} params")
print(f"Torch   : {device_torch}")

# ============================================================================
# Section 2 — Seeds
# ============================================================================

torch.set_default_dtype(torch.float32)
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

# ============================================================================
# Section 3 — Pauli Basis (generic for any n_qubits)
# ============================================================================

norm_factor = 1.0 / dim
I_m = np.array([[1, 0], [0, 1]],   dtype=complex)
X_m = np.array([[0, 1], [1, 0]],   dtype=complex)
Y_m = np.array([[0,-1j],[1j, 0]],  dtype=complex)
Z_m = np.array([[1, 0], [0,-1]],   dtype=complex)
_paulis_1q = [I_m, X_m, Y_m, Z_m]
_labels_1q = ['I', 'X', 'Y', 'Z']

full_paulis, full_labels = [], []
for index in range(4 ** n_qubits):
    digits, tmp = [], index
    for _ in range(n_qubits):
        digits.append(tmp % 4); tmp //= 4
    digits = digits[::-1]
    P   = _paulis_1q[digits[0]]
    lab = _labels_1q[digits[0]]
    for d in digits[1:]:
        P   = np.kron(P, _paulis_1q[d])
        lab = lab + _labels_1q[d]
    full_paulis.append(P)
    full_labels.append(lab)

identity_label   = 'I' * n_qubits
pauli_idx_nonid  = [i for i, l in enumerate(full_labels) if l != identity_label]
pauli_idx_ident  = [i for i, l in enumerate(full_labels) if l == identity_label][0]

full_paulis_arr  = np.stack(full_paulis, axis=0)                              # (4^n, dim, dim)
pauli_torch      = torch.tensor(full_paulis_arr, dtype=torch.complex64,
                                device=device_torch)                          # (4^n, dim, dim)
nonid_idx_torch  = torch.tensor(pauli_idx_nonid, dtype=torch.long,
                                device=device_torch)                          # (n_pauli_nonid,)

assert len(pauli_idx_nonid) == n_pauli_nonid
print(f"Pauli basis: {4**n_qubits} total, {n_pauli_nonid} non-identity")

# ============================================================================
# Section 4 — Classical Helper Functions
# ============================================================================

def validate_density_matrix(rho, tol=1e-6):
    rho = (rho + rho.conj().T) / 2.0
    tr  = np.trace(rho)
    if abs(tr) > 1e-12:
        rho = rho / tr
    evals, evecs = np.linalg.eigh(rho)
    if np.any(evals < -tol):
        evals = np.maximum(evals, 0.0)
        rho   = evecs @ np.diag(evals) @ evecs.conj().T
        rho   = rho / np.trace(rho)
    return rho

def compute_purity(rho):
    return float(np.real(np.trace(rho @ rho)))

def random_haar_statevector(n_q):
    d   = 2 ** n_q
    vec = np.random.randn(d) + 1j * np.random.randn(d)
    return vec / np.linalg.norm(vec)

def rho_to_pauli_exps(rho):
    """Density matrix → 63-dim float32 Pauli expectation vector."""
    exps = []
    for idx in pauli_idx_nonid:
        exps.append(float(np.real_if_close(np.trace(rho @ full_paulis_arr[idx]))))
    return np.array(exps, dtype=np.float32)

def pauli_exps_to_rho(exps_nonid):
    """63-dim Pauli vector → 8×8 complex density matrix (numpy)."""
    exps_full = np.zeros(4 ** n_qubits, dtype=float)
    exps_full[pauli_idx_ident] = 1.0
    for i, idx in enumerate(pauli_idx_nonid):
        exps_full[idx] = float(exps_nonid[i])
    rho = sum(c * P for c, P in zip(exps_full, full_paulis_arr))
    rho = rho * norm_factor
    return (rho + rho.conj().T) / 2.0

def fidelity_numpy(rho_true, rho_pred):
    """Uhlmann-Jozsa fidelity (numpy, used only in shot-noise evaluation)."""
    eps  = 1e-12
    rt   = rho_true + eps * np.eye(dim)
    rp   = rho_pred + eps * np.eye(dim)
    sq   = sqrtm(rt)
    tr   = np.real(np.trace(sqrtm(sq @ rp @ sq)))
    return float(np.clip(tr ** 2, 0.0, 1.0))

def purity_to_depol_param(target_purity, n_q):
    """Solve (1-p)² + p(2-p)/d = target_purity for p."""
    if target_purity >= 0.99:
        return 0.0
    d = 2 ** n_q
    a = 1.0 + 1.0 / d
    b = -2.0
    c = 1.0 - target_purity
    disc = b ** 2 - 4 * a * c
    if disc < 0:
        return 0.0
    return float(np.clip((-b - np.sqrt(disc)) / (2 * a), 0.0, 1.0))

# ============================================================================
# Section 5 — Mixed State Generators
# ============================================================================

def _amplitude_damped(gamma):
    psi = random_haar_statevector(n_qubits)
    rho = np.outer(psi, psi.conj())
    K0  = np.array([[1, 0], [0, np.sqrt(1 - gamma)]], dtype=complex)
    K1  = np.array([[0, np.sqrt(gamma)], [0, 0]],     dtype=complex)
    for q in range(n_qubits):
        parts0 = [K0 if i == q else np.eye(2, dtype=complex) for i in range(n_qubits)]
        parts1 = [K1 if i == q else np.eye(2, dtype=complex) for i in range(n_qubits)]
        K0f = parts0[0];  K1f = parts1[0]
        for p0, p1 in zip(parts0[1:], parts1[1:]):
            K0f = np.kron(K0f, p0);  K1f = np.kron(K1f, p1)
        rho = K0f @ rho @ K0f.conj().T + K1f @ rho @ K1f.conj().T
    return validate_density_matrix(rho)

def _phase_damped(gamma):
    psi = random_haar_statevector(n_qubits)
    rho = np.outer(psi, psi.conj())
    K0 = np.sqrt(1 - gamma) * np.eye(2, dtype=complex)
    K1 = np.sqrt(gamma)     * np.array([[1,0],[0,0]], dtype=complex)
    K2 = np.sqrt(gamma)     * np.array([[0,0],[0,1]], dtype=complex)
    for q in range(n_qubits):
        rho_new = np.zeros_like(rho)
        for K1q in [K0, K1, K2]:
            parts = [K1q if i == q else np.eye(2, dtype=complex) for i in range(n_qubits)]
            Kf = parts[0]
            for kp in parts[1:]:
                Kf = np.kron(Kf, kp)
            rho_new += Kf @ rho @ Kf.conj().T
        rho = rho_new
    return validate_density_matrix(rho)

def _thermal(beta):
    H = np.random.randn(dim, dim) + 1j * np.random.randn(dim, dim)
    H = (H + H.conj().T) / 2.0
    evals, evecs = np.linalg.eigh(H)
    evals -= evals[0]
    H     = evecs @ np.diag(evals) @ evecs.conj().T
    expH  = scipy.linalg.expm(-beta * H)
    return validate_density_matrix(expH / np.trace(expH))

def _make_generators(min_p, max_p):
    """Return list of (name, callable) state generators with purity control."""

    def _sp():
        return np.random.uniform(min_p, max_p)

    def depolarized():
        p   = purity_to_depol_param(_sp(), n_qubits)
        psi = random_haar_statevector(n_qubits)
        rho = np.outer(psi, psi.conj())
        return validate_density_matrix((1-p)*rho + p*np.eye(dim, dtype=complex)/dim)

    def werner():
        tp  = _sp()
        d   = dim
        a   = 1+1/d; bw = -2/d; cw = 1/d - tp
        disc = bw**2 - 4*a*cw
        p   = np.clip((-bw + np.sqrt(max(disc, 0)))/(2*a), 0, 1) if disc >= 0 else 0.5
        psi = np.zeros(dim, dtype=complex); psi[0] = psi[-1] = 1/np.sqrt(2)
        pr  = np.outer(psi, psi.conj())
        return validate_density_matrix(p*pr + (1-p)*np.eye(dim, dtype=complex)/dim)

    def isotropic():
        p   = purity_to_depol_param(_sp(), n_qubits)
        psi = random_haar_statevector(n_qubits)
        pr  = np.outer(psi, psi.conj())
        return validate_density_matrix((1-p)*pr + p*np.eye(dim, dtype=complex)/dim)

    def _bisect(fn, target, lo, hi, flipped=False, iters=20):
        """Binary search over channel param to hit target purity."""
        for _ in range(iters):
            mid = (lo + hi) / 2
            pur = compute_purity(fn(mid))
            if (pur > target) ^ flipped:
                lo = mid
            else:
                hi = mid
        return fn((lo + hi) / 2)

    def amp_damped():
        return _bisect(_amplitude_damped, _sp(), 0.0, 0.99, flipped=False)

    def phase_damped():
        return _bisect(_phase_damped, _sp(), 0.0, 0.99, flipped=False)

    def thermal():
        return _bisect(_thermal, _sp(), 0.01, 10.0, flipped=True)

    def separable():
        tp  = _sp()
        pq  = tp ** (1.0 / n_qubits)
        rho = None
        for _ in range(n_qubits):
            p_this = float(np.clip(np.random.uniform(pq-0.03, pq+0.03), 0.5, 1.0))
            p_dep  = purity_to_depol_param(p_this, 1)
            psi_q  = random_haar_statevector(1)
            rq     = np.outer(psi_q, psi_q.conj())
            rq     = (1-p_dep)*rq + p_dep*np.eye(2, dtype=complex)/2
            rho    = rq if rho is None else np.kron(rho, rq)
        return validate_density_matrix(rho)

    return [
        ('Depolarized', depolarized),
        ('Werner',      werner),
        ('Isotropic',   isotropic),
        ('AmpDamped',   amp_damped),
        ('PhaseDamped', phase_damped),
        ('Thermal',     thermal),
        ('Separable',   separable),
    ]

def _gen_one(gen_fn, min_p, max_p, max_tries=120):
    """Generate a single valid density matrix. Used as joblib task."""
    for _ in range(max_tries):
        try:
            rho = gen_fn()
            p   = compute_purity(rho)
            if min_p - 0.01 <= p <= max_p + 0.01:
                return rho_to_pauli_exps(rho), rho.astype(np.complex64)
        except Exception:
            pass
    return None

def build_dataset(n_samples, min_purity=0.85, max_purity=0.95):
    generators = _make_generators(min_purity, max_purity)
    n_per_type = n_samples // len(generators)
    remainder  = n_samples %  len(generators)
    X_list, Rho_list = [], []

    for gi, (name, fn) in enumerate(generators):
        n_this = n_per_type + (1 if gi < remainder else 0)
        over   = n_this * 4   # over-generate to ensure enough valid samples
        print(f"  {name}: requesting {n_this} ...", end='', flush=True)

        if HAS_JOBLIB and N_JOBS_DATA > 1:
            results = Parallel(n_jobs=N_JOBS_DATA)(
                delayed(_gen_one)(fn, min_purity, max_purity) for _ in range(over))
        else:
            results = [_gen_one(fn, min_purity, max_purity) for _ in range(over)]

        valid = [r for r in results if r is not None][:n_this]
        if len(valid) < n_this:
            print(f" WARNING only {len(valid)}/{n_this}!", end='')

        for exps, rho in valid:
            X_list.append(exps)
            Rho_list.append(rho)
        print(f" got {len(valid)}")

    X    = np.stack(X_list)
    Rhos = np.stack(Rho_list)
    purs = [compute_purity(r) for r in Rhos]
    print(f"  Total {X.shape[0]} | purity [{min(purs):.3f}, {max(purs):.3f}] "
          f"mean={np.mean(purs):.3f}")
    return X, Rhos

# ============================================================================
# Section 6 — Dataset (Load Cache or Generate)
# ============================================================================

if os.path.exists(DATASET_CACHE):
    print(f"Loading cached dataset: {DATASET_CACHE}")
    _d = np.load(DATASET_CACHE)
    X_train_np, Rho_train_np = _d['X_train'], _d['Rho_train']
    X_val_np,   Rho_val_np   = _d['X_val'],   _d['Rho_val']
    print(f"  Train {X_train_np.shape} | Val {X_val_np.shape}")
else:
    print("Generating training dataset...")
    X_train_np, Rho_train_np = build_dataset(n_train)
    print("Generating validation dataset...")
    X_val_np,   Rho_val_np   = build_dataset(n_val)
    np.savez_compressed(DATASET_CACHE,
                        X_train=X_train_np,   Rho_train=Rho_train_np,
                        X_val=X_val_np,       Rho_val=Rho_val_np)
    print(f"Dataset saved to {DATASET_CACHE}")

X_train   = torch.tensor(X_train_np,   dtype=torch.float32,   device=device_torch)
X_val     = torch.tensor(X_val_np,     dtype=torch.float32,   device=device_torch)
Rho_train = torch.tensor(Rho_train_np, dtype=torch.complex64, device=device_torch)
Rho_val   = torch.tensor(Rho_val_np,   dtype=torch.complex64, device=device_torch)

train_loader = DataLoader(TensorDataset(X_train, Rho_train),
                          batch_size=batch_size, shuffle=True,  drop_last=False)
val_loader   = DataLoader(TensorDataset(X_val,   Rho_val),
                          batch_size=batch_size, shuffle=False)

# ============================================================================
# Section 7 — PennyLane Device + Mixed-State Decoder Circuit
# ============================================================================
#
# WHY default.mixed IS NOW LEGITIMATE:
#   The decoder inserts qml.DepolarizingChannel after every unitary layer.
#   Noise channels require density-matrix simulation — default.mixed is the
#   correct device. The output is a genuinely mixed state for any non-zero p.
#
# WHY diff_method='backprop':
#   The adjoint method only supports unitary circuits. Once noise channels are
#   present, backprop is the appropriate (and PennyLane-supported) choice.
#
# INITIAL STATE:
#   PennyLane default.mixed initialises to |0...0><0...0| when no explicit
#   state preparation is specified. After unitary + noise layers, the output
#   is a mixed state. (The paper's claim of rho_0 = I/d was erroneous.)

dev = qml.device("default.mixed", wires=n_qubits, shots=shots)


@qml.qnode(dev, interface='torch', diff_method='backprop')
def circuit_pauli_exps(params):
    """
    Mixed-state decoder circuit.

    Args:
        params : 1-D tensor of length n_circuit_params = n_angles + n_noise_params
                 The noise entries are already scaled to [0, NOISE_MAX] by the model.

    For each layer ℓ:
        1.  RX / RY / RZ on each qubit          (parameterised unitary)
        2.  CNOT chain  [0→1, 1→2]              (entanglement)
        3.  DepolarizingChannel(p_ℓ) on each q  (learned noise → mixed output)

    Returns tuple of 63 Pauli expectation values.
    """
    angles  = params[:n_angles]
    noise_p = params[n_angles:]          # length n_noise_params, ∈ [0, NOISE_MAX]

    for layer in range(n_layers):
        # --- Single-qubit rotations ---
        for w in range(n_qubits):
            base = (layer * n_qubits + w) * 3
            qml.RX(angles[base],     wires=w)
            qml.RY(angles[base + 1], wires=w)
            qml.RZ(angles[base + 2], wires=w)

        # --- Entangling layer (linear topology) ---
        for w in range(n_qubits - 1):
            qml.CNOT(wires=[w, w + 1])

        # --- Learnable depolarising noise (one param per layer, shared across qubits) ---
        p_layer = noise_p[layer]
        for w in range(n_qubits):
            qml.DepolarizingChannel(p_layer, wires=w)

    # --- Measure all 63 non-identity Pauli expectations ---
    exps = []
    for idx_label in pauli_idx_nonid:
        lab = full_labels[idx_label]
        ops = []
        for wire, ch in enumerate(lab):
            if   ch == 'X': ops.append(qml.PauliX(wires=wire))
            elif ch == 'Y': ops.append(qml.PauliY(wires=wire))
            elif ch == 'Z': ops.append(qml.PauliZ(wires=wire))
            else:           ops.append(qml.Identity(wires=wire))
        obs = ops[0]
        for op in ops[1:]:
            obs = obs @ op
        exps.append(qml.expval(obs))
    return tuple(exps)


# --- Try batched vmap execution ---
try:
    _test_p = torch.zeros(2, n_circuit_params, device=device_torch)
    batched_circuit = qml.vmap(circuit_pauli_exps, in_axes=0, out_axes=0)
    _  = batched_circuit(_test_p)          # smoke-test
    vmap_available = True
    print("qml.vmap: enabled (batched circuit execution)")
except Exception as e:
    batched_circuit = None
    vmap_available  = False
    print(f"qml.vmap: disabled ({e}) — using per-sample fallback")

# ============================================================================
# Section 8 — Model Architecture
# ============================================================================

class Encoder(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Linear(128, latent_dim),
        )

    def forward(self, x):
        return self.net(x)


class HybridAE(nn.Module):
    """
    Hybrid Quantum-Classical Autoencoder with Mixed-State Decoder.

    Encoder  : classical MLP  (63 → 256 → 128 → 20)
    Decoder  : PQC + DepolarizingChannel  (20 → 60 circuit params → mixed ρ_pred)

    The latent vector z is mapped to:
      • rotation angles  (latent2angles, 54 params)
      • per-layer noise  (latent2noise, 6 params, sigmoid-scaled to [0, NOISE_MAX])
    Both are concatenated into a single params vector fed to circuit_pauli_exps.
    """

    def __init__(self, input_dim, latent_dim, n_ang, n_noise, vmap_fn=None):
        super().__init__()
        self.encoder       = Encoder(input_dim, latent_dim)
        self.latent2angles = nn.Linear(latent_dim, n_ang)
        self.latent2noise  = nn.Linear(latent_dim, n_noise)
        self.n_ang         = n_ang
        self.n_noise       = n_noise
        self.vmap_fn       = vmap_fn

        # Initialise noise biases so initial noise ≈ 0.011 per layer
        # sigmoid(-2.5) * NOISE_MAX = 0.076 * 0.15 ≈ 0.011
        nn.init.constant_(self.latent2noise.bias, -2.5)

    def forward(self, x):
        z          = self.encoder(x)                                          # (B, latent_dim)
        angles     = self.latent2angles(z)                                    # (B, n_ang)
        noise_p    = torch.sigmoid(self.latent2noise(z)) * NOISE_MAX          # (B, n_noise)
        params     = torch.cat([angles, noise_p], dim=-1)                     # (B, n_circuit_params)
        B          = params.shape[0]

        if self.vmap_fn is not None and B > 0:
            out = self.vmap_fn(params)
            if isinstance(out, (tuple, list)):
                preds = torch.stack([o.to(x.dtype) for o in out], dim=-1)
            else:
                preds = out.to(x.dtype)
            if preds.dim() == 2 and preds.shape[0] != B:
                preds = preds.transpose(0, 1)
        else:
            preds_list = []
            for i in range(B):
                res = circuit_pauli_exps(params[i])
                preds_list.append(torch.stack(list(res)))
            preds = torch.stack(preds_list, dim=0).to(x.dtype)

        return preds, z

    @torch.no_grad()
    def get_noise_params(self, x):
        """Return learned per-layer noise strengths for an input batch."""
        z = self.encoder(x)
        return torch.sigmoid(self.latent2noise(z)) * NOISE_MAX   # (B, n_noise)


model = HybridAE(
    input_dim  = n_pauli_nonid,
    latent_dim = latent_dim,
    n_ang      = n_angles,
    n_noise    = n_noise_params,
    vmap_fn    = batched_circuit if vmap_available else None,
).to(device_torch)

n_params_total = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model trainable parameters: {n_params_total:,}")

optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, factor=0.5, patience=12, min_lr=1e-5)

# ============================================================================
# Section 9 — Loss Functions (Fully GPU-Batched)
# ============================================================================

def pauli_exps_to_rho_torch(exps_nonid):
    """
    Vectorised reconstruction: (B, 63) Pauli exps → (B, 8, 8) density matrices.
    Uses advanced indexing — no Python loop over Paulis.
    """
    B        = exps_nonid.shape[0]
    exps_full = torch.zeros(B, 4**n_qubits,
                            dtype=torch.float32, device=exps_nonid.device)
    exps_full[:, pauli_idx_ident]    = 1.0
    exps_full[:, nonid_idx_torch]    = exps_nonid          # vectorised scatter
    exps_c   = exps_full.to(torch.complex64).unsqueeze(-1).unsqueeze(-1)  # (B,4^n,1,1)
    rho      = (exps_c * pauli_torch.unsqueeze(0)).sum(dim=1) * norm_factor  # (B,dim,dim)
    return 0.5 * (rho + rho.conj().permute(0, 2, 1))


def sqrtm_psd_torch(A, eps=1e-12):
    """Batched matrix square root for PSD matrices via eigendecomposition."""
    A    = 0.5 * (A + A.conj().transpose(-1, -2))
    A    = A + eps * torch.eye(A.shape[-1], dtype=A.dtype, device=A.device)
    evals, evecs = torch.linalg.eigh(A)
    evals = torch.clamp(evals, min=0.0)
    return (evecs * torch.sqrt(evals).unsqueeze(-2)) @ evecs.conj().transpose(-1, -2)


def uhlmann_fidelity_batch(rho_a, rho_b, eps=1e-12):
    """
    Batched Uhlmann-Jozsa fidelity: F(ρ,σ) = (Tr√(√ρ σ √ρ))²
    Args:
        rho_a, rho_b: (B, dim, dim) complex64
    Returns:
        F: (B,) float, ∈ [0, 1]
    """
    rho_a   = 0.5 * (rho_a + rho_a.conj().transpose(-1, -2))
    rho_b   = 0.5 * (rho_b + rho_b.conj().transpose(-1, -2))
    sqrt_a  = sqrtm_psd_torch(rho_a, eps)
    M       = sqrt_a @ rho_b @ sqrt_a
    M       = 0.5 * (M + M.conj().transpose(-1, -2))
    evals, _ = torch.linalg.eigh(M)
    tr_sqrt = torch.sum(torch.sqrt(torch.clamp(evals, min=0.0) + eps), dim=-1)
    return torch.clamp(tr_sqrt ** 2, 0.0, 1.0).real


def metric_preservation_loss(z_batch, rho_batch, n_pairs=50):
    """
    Vectorised metric-preservation loss.
    Encourages: d_latent(z_i, z_j) ∝ d_Bures(ρ_i, ρ_j)

    Args:
        z_batch   : (B, latent_dim) — latent vectors
        rho_batch : (B, dim, dim) complex64 — *true* density matrices
        n_pairs   : random pairs to sample per call
    Returns: scalar loss
    """
    B = z_batch.shape[0]
    if B < 2:
        return torch.tensor(0.0, device=z_batch.device)

    n_pairs = min(n_pairs, B * (B - 1) // 2)
    idx_i   = torch.randint(0, B, (n_pairs,), device=z_batch.device)
    offsets = torch.randint(1, B, (n_pairs,), device=z_batch.device)
    idx_j   = (idx_i + offsets) % B

    d_latent = torch.norm(z_batch[idx_i] - z_batch[idx_j], dim=-1)       # (n_pairs,)
    F        = uhlmann_fidelity_batch(rho_batch[idx_i], rho_batch[idx_j]) # (n_pairs,)
    d_bures  = torch.arccos(
        torch.clamp(torch.sqrt(torch.clamp(F, 0.0, 1.0)), 0.0, 1.0 - 1e-7))

    valid  = d_bures > 1e-6
    if not valid.any():
        return torch.tensor(0.0, device=z_batch.device)

    ratio = d_latent[valid] / (d_bures[valid] + 1e-8)
    return ((ratio - 1.0) ** 2).mean()


def evaluate_fidelity(model, X_np, Rho_np):
    """
    Batched GPU fidelity evaluation — no scipy sqrtm loop.
    Replaces the original O(N_val) serial scipy calls with vectorised torch ops.
    """
    model.eval()
    pred_chunks = []
    with torch.no_grad():
        for i in range(0, len(X_np), batch_size):
            xb      = torch.tensor(X_np[i:i+batch_size],
                                   dtype=torch.float32, device=device_torch)
            preds_b, _ = model(xb)
            pred_chunks.append(preds_b)

    preds_all    = torch.cat(pred_chunks, dim=0)
    rho_pred_all = pauli_exps_to_rho_torch(preds_all)
    rho_true_all = torch.tensor(Rho_np, dtype=torch.complex64, device=device_torch)

    # Process in chunks to avoid OOM
    fids = []
    for i in range(0, len(preds_all), batch_size):
        f = uhlmann_fidelity_batch(rho_pred_all[i:i+batch_size],
                                   rho_true_all[i:i+batch_size])
        fids.append(f.cpu())

    fids = torch.cat(fids).numpy()
    return float(fids.mean()), float(np.median(fids)), fids.tolist()

# ============================================================================
# Section 10 — Training Loop
# ============================================================================

history = dict(epoch=[], loss_total=[], loss_recon=[],
               loss_metric=[], val_fid=[], noise_mean=[])

best_val_fid    = 0.0
best_state      = None
no_improve_evals = 0
t0              = time.time()

print("\n" + "=" * 72)
print("TRAINING — 3-Qubit Hybrid Autoencoder (Mixed-State Decoder)")
print(f"  λ={lambda_metric}  NOISE_MAX={NOISE_MAX}  eval_every={eval_every}")
print("=" * 72)

for epoch in range(1, max_epochs + 1):
    model.train()
    run_loss = run_recon = run_metric = 0.0
    iters    = 0

    for x_batch, rho_true_batch in train_loader:
        x_batch        = x_batch.to(device_torch)
        rho_true_batch = rho_true_batch.to(device_torch)
        optimizer.zero_grad()

        preds, z = model(x_batch)

        # --- Reconstruction loss (Uhlmann fidelity) ---
        if use_uhlmann_loss:
            rho_pred   = pauli_exps_to_rho_torch(preds)
            F_batch    = uhlmann_fidelity_batch(rho_pred, rho_true_batch)
            loss_recon = 1.0 - F_batch.mean()
        else:
            loss_recon = nn.functional.mse_loss(preds, x_batch)

        # --- Metric preservation loss (uses TRUE rho for stable Bures distances) ---
        loss_metric = metric_preservation_loss(z, rho_true_batch, n_pairs=50)

        loss = loss_recon + lambda_metric * loss_metric
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
        optimizer.step()

        run_loss   += loss.item()
        run_recon  += loss_recon.item()
        run_metric += loss_metric.item()
        iters      += 1

    avg_loss   = run_loss   / max(1, iters)
    avg_recon  = run_recon  / max(1, iters)
    avg_metric = run_metric / max(1, iters)

    # --- Periodic fidelity evaluation ---
    if epoch % eval_every == 0 or epoch == 1:
        val_mean, val_median, _ = evaluate_fidelity(model, X_val_np, Rho_val_np)

        with torch.no_grad():
            noise_vals = model.get_noise_params(X_train[:64])
            noise_mean = noise_vals.mean().item()

        history['epoch'].append(epoch)
        history['loss_total'].append(avg_loss)
        history['loss_recon'].append(avg_recon)
        history['loss_metric'].append(avg_metric)
        history['val_fid'].append(val_mean)
        history['noise_mean'].append(noise_mean)

        scheduler.step(1.0 - val_mean)

        elapsed = time.time() - t0
        print(f"Ep {epoch:03d} | loss {avg_loss:.5f} "
              f"(recon {avg_recon:.5f} | metric {avg_metric:.5f}) "
              f"| val F {val_mean:.4f} | noise {noise_mean:.4f} | {elapsed:.0f}s")

        if val_mean > best_val_fid:
            best_val_fid = val_mean
            best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve_evals = 0
        else:
            no_improve_evals += 1

        if no_improve_evals >= early_stop_patience:
            print(f"Early stopping at epoch {epoch} "
                  f"({no_improve_evals} eval steps without improvement)")
            break

        if val_mean >= target_fidelity:
            print(f"Target fidelity {target_fidelity} reached at epoch {epoch}!")
            break

if best_state is not None:
    model.load_state_dict(best_state)
    print(f"\nRestored best model: val fidelity = {best_val_fid:.4f}")

with open(os.path.join(OUTDIR, 'training_history_3q.pkl'), 'wb') as f:
    pickle.dump(history, f)

val_mean_fid, val_median_fid, all_fids = evaluate_fidelity(model, X_val_np, Rho_val_np)
print(f"Final val: mean={val_mean_fid:.4f} | median={val_median_fid:.4f}")

# --- Report learned noise parameters ---
model.eval()
with torch.no_grad():
    noise_report = model.get_noise_params(X_val[:200])
print("\nLearned depolarising noise per layer:")
for l in range(n_noise_params):
    v = noise_report[:, l].cpu().numpy()
    print(f"  Layer {l+1}: {v.mean():.4f} ± {v.std():.4f}")

# ============================================================================
# Section 11 — Shot Noise Robustness Evaluation
# ============================================================================
#
# Strategy: fast analytical variance injection.
#   The shot-noise variance for a Pauli expectation ⟨P⟩ from S shots is
#       Var(⟨P⟩_S) = (1 - ⟨P⟩²) / S
#   We add Gaussian noise with this variance to the analytical predictions,
#   then measure how reconstruction fidelity degrades.
#   This is exact in the large-S limit and far faster than re-running PL circuits.

print("\n" + "=" * 72)
print("SHOT NOISE ROBUSTNESS EVALUATION")
print("=" * 72)

def evaluate_shot_noise(model, X_np, Rho_np, shots_count, n_eval=N_EVAL_SHOTS):
    """
    Evaluate fidelity under finite-shot measurement noise.
    Adds Gaussian noise with variance = (1 - <P>²) / shots to each expectation value.
    """
    model.eval()
    idx = np.random.choice(len(X_np), min(n_eval, len(X_np)), replace=False)

    with torch.no_grad():
        xb          = torch.tensor(X_np[idx], dtype=torch.float32, device=device_torch)
        preds_b, _  = model(xb)
        preds_np    = preds_b.cpu().numpy()

    fids = []
    for i, (pred_analyt, true_idx) in enumerate(zip(preds_np, idx)):
        variance    = (1.0 - pred_analyt ** 2) / shots_count
        noise       = np.random.randn(len(pred_analyt)) * np.sqrt(np.maximum(variance, 0.0))
        pred_noisy  = np.clip(pred_analyt + noise, -1.0, 1.0)
        rho_pred    = pauli_exps_to_rho(pred_noisy)
        rho_true    = Rho_np[true_idx]
        fids.append(fidelity_numpy(rho_true, rho_pred))

    return float(np.mean(fids)), float(np.std(fids))

shot_results = {}
for sv in SHOT_COUNTS:
    label = 'Analytical (∞ shots)' if sv is None else f'{sv:,} shots'
    if sv is None:
        mean_f, std_f = val_mean_fid, float(np.std(all_fids))
    else:
        print(f"  {label} ...", end='', flush=True)
        mean_f, std_f = evaluate_shot_noise(model, X_val_np, Rho_val_np, sv)
        print(f"  F = {mean_f:.4f} ± {std_f:.4f}")
    shot_results[label] = (mean_f, std_f)

print("\nShot Noise Summary:")
print(f"{'Condition':>25} | {'Mean F':>8} | {'Std':>7}")
print("-" * 50)
for lbl, (mf, sf) in shot_results.items():
    print(f"{lbl:>25} | {mf:>8.4f} | {sf:>7.4f}")

# ============================================================================
# Section 12 — Manifold Geometry Analysis
# ============================================================================

print("\n" + "=" * 72)
print("MANIFOLD GEOMETRY ANALYSIS")
print("=" * 72)

# --- Extract latent vectors for the full dataset ---
model.eval()
with torch.no_grad():
    X_all       = torch.cat([X_train, X_val], dim=0)
    Rho_all_t   = torch.cat([Rho_train, Rho_val], dim=0)
    z_chunks    = [model.encoder(X_all[i:i+batch_size]).cpu().numpy()
                   for i in range(0, len(X_all), batch_size)]
    Z           = np.vstack(z_chunks)
    Rho_all_np  = Rho_all_t.cpu().numpy()

print(f"Latent matrix Z: {Z.shape}")

# --- Geodesic-Bures correlation (GPU batched) ---
print(f"Computing geodesic-Bures correlation ({n_geodesic_pairs} pairs, GPU) ...")
np.random.seed(seed)
pair_idx    = np.array([np.random.choice(len(Z), 2, replace=False)
                        for _ in range(n_geodesic_pairs)])
ii, jj      = pair_idx[:, 0], pair_idx[:, 1]
geo_dists   = np.linalg.norm(Z[ii] - Z[jj], axis=1)

Ri = torch.tensor(Rho_all_np[ii], dtype=torch.complex64, device=device_torch)
Rj = torch.tensor(Rho_all_np[jj], dtype=torch.complex64, device=device_torch)
F_chunks = []
for k in range(0, len(ii), 50):
    F_chunks.append(uhlmann_fidelity_batch(Ri[k:k+50], Rj[k:k+50]).cpu().numpy())
F_vals = np.concatenate(F_chunks)

bures_dists = np.arccos(np.sqrt(np.clip(F_vals, 0.0, 1.0)))
valid_mask  = bures_dists > 1e-6
geo_dists   = geo_dists[valid_mask]
bures_dists = bures_dists[valid_mask]

pearson_r,  pearson_p  = pearsonr(geo_dists,   bures_dists)
spearman_r, spearman_p = spearmanr(geo_dists,  bures_dists)
coeffs   = np.polyfit(bures_dists, geo_dists, 1)
a_fit, b_fit = coeffs[0], coeffs[1]
geo_pred = a_fit * bures_dists + b_fit
r_squared = 1 - np.sum((geo_dists - geo_pred)**2) / np.sum((geo_dists - geo_dists.mean())**2)

print(f"  Pearson  r  = {pearson_r:.4f}  (p={pearson_p:.2e})")
print(f"  Spearman ρ  = {spearman_r:.4f}  (p={spearman_p:.2e})")
print(f"  R²          = {r_squared:.4f}")
print(f"  Linear fit  : d_L = {a_fit:.4f}·d_B + {b_fit:.4f}")

# --- Intrinsic dimensionality ---
def mle_intrinsic_dim(X, k=15):
    nbrs  = NearestNeighbors(n_neighbors=k+1).fit(X)
    dists, _ = nbrs.kneighbors(X)
    dists = dists[:, 1:]
    dims  = []
    for row in dists:
        r_max = row[-1]
        if r_max > 1e-10:
            dims.append(-k / np.sum(np.log(row / r_max + 1e-12)))
    return np.mean(dims), np.std(dims)

id_mean, id_std = mle_intrinsic_dim(Z, k=min(15, Z.shape[0]//4))
pca_full        = PCA().fit(Z)
cumvar          = np.cumsum(pca_full.explained_variance_ratio_)
dim_95 = int(np.argmax(cumvar >= 0.95)) + 1
dim_99 = int(np.argmax(cumvar >= 0.99)) + 1
print(f"\nIntrinsic dim (MLE): {id_mean:.2f} ± {id_std:.2f}")
print(f"PCA 95%: {dim_95} dims | PCA 99%: {dim_99} dims")
print(f"Top-5 eigenvalues: {pca_full.explained_variance_ratio_[:5]}")

# --- Local curvature ---
def local_curvature(X, k=25):
    nbrs    = NearestNeighbors(n_neighbors=k+1).fit(X)
    _, idxs = nbrs.kneighbors(X)
    kappas  = []
    for i in range(X.shape[0]):
        nb = X[idxs[i, 1:]] - X[i]
        if nb.shape[0] > 1:
            try:
                _, s, _ = np.linalg.svd(nb, full_matrices=False)
                s = s[s > 1e-10]
                if len(s) > 1:
                    kappas.append(s[-1] / s[0])
            except Exception:
                pass
    return np.array(kappas) if kappas else np.array([0.0])

k_curv   = min(25, Z.shape[0] // 4)
curvatures = local_curvature(Z, k=k_curv)
print(f"\nCurvature (k={k_curv}): "
      f"mean={np.mean(curvatures):.5f} ± {np.std(curvatures):.5f}  "
      f"IQR=[{np.percentile(curvatures,25):.5f}, {np.percentile(curvatures,75):.5f}]")

# --- Bures distance matrix for visualisation (GPU batched, no scipy loop) ---
n_vis = min(n_vis_samples, Z.shape[0])
np.random.seed(seed + 1)
vis_idx     = np.random.choice(Z.shape[0], n_vis, replace=False)
Z_vis       = Z[vis_idx]
Rho_vis_np  = Rho_all_np[vis_idx]

print(f"\nComputing {n_vis}×{n_vis} Bures matrix (GPU batched) ...")
Rho_vis_t   = torch.tensor(Rho_vis_np, dtype=torch.complex64, device=device_torch)
bures_mat   = np.zeros((n_vis, n_vis))
chunk_sz    = 15   # conservative for 8×8 density matrices

for ci_start in range(0, n_vis, chunk_sz):
    ci_end = min(ci_start + chunk_sz, n_vis)
    ci     = ci_end - ci_start
    ri     = Rho_vis_t[ci_start:ci_end].unsqueeze(1).expand(-1, n_vis, -1, -1).reshape(-1, dim, dim)
    rj     = Rho_vis_t.unsqueeze(0).expand(ci, -1, -1, -1).reshape(-1, dim, dim)
    F_flat = uhlmann_fidelity_batch(ri, rj).cpu().numpy()
    bures_mat[ci_start:ci_end, :] = np.arccos(np.sqrt(np.clip(F_flat.reshape(ci, n_vis), 0, 1)))

bures_mat = (bures_mat + bures_mat.T) / 2
np.fill_diagonal(bures_mat, 0.0)

# --- MDS and PCA embeddings for figures ---
print("Running MDS (3D + 2D) ...")
mds3   = MDS(n_components=3, dissimilarity='precomputed', random_state=seed, normalized_stress='auto')
q3d    = mds3.fit_transform(bures_mat)
mds2   = MDS(n_components=2, dissimilarity='precomputed', random_state=seed, normalized_stress='auto')
q2d    = mds2.fit_transform(bures_mat)
pca3_v = PCA(n_components=3, random_state=seed).fit_transform(Z_vis)
pca2_v = PCA(n_components=2, random_state=seed).fit_transform(Z_vis)

R_proc, _  = orthogonal_procrustes(pca3_v, q3d)
lat_aligned = pca3_v @ R_proc
rmse_proc  = float(np.sqrt(np.mean((q3d - lat_aligned) ** 2)))
print(f"Procrustes RMSE: {rmse_proc:.4f}")

# ============================================================================
# Section 13 — Figures
# ============================================================================

colors    = np.arange(n_vis)
bures_line = np.linspace(bures_dists.min(), bures_dists.max(), 200)

# ── Figure 1: Main result — geodesic-Bures correlation ───────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sc = axes[0].scatter(bures_dists[:400], geo_dists[:400],
                     c=geo_dists[:400], cmap='plasma', s=20, alpha=0.6, rasterized=True)
axes[0].plot(bures_line, a_fit*bures_line + b_fit, 'r-', lw=2.5,
             label=f'$d_L = {a_fit:.3f}d_B + {b_fit:.3f}$\n$R^2={r_squared:.4f}$')
axes[0].set_xlabel('Bures Distance $d_B$', fontweight='bold')
axes[0].set_ylabel('Latent Euclidean Distance $d_L$', fontweight='bold')
axes[0].set_title(f'(A) Geodesic-Bures Correlation  r = {pearson_r:.4f}',
                  fontweight='bold', loc='left')
axes[0].legend(); axes[0].grid(True, alpha=0.3)
plt.colorbar(sc, ax=axes[0])

F_th = np.linspace(0.01, 1.0, 200)
axes[1].plot(F_th, np.arccos(np.sqrt(F_th)), 'b-', lw=2.5,
             label='$D_B = \\arccos(\\sqrt{F})$')
axes[1].set_xlabel('Fidelity $F$', fontweight='bold')
axes[1].set_ylabel('Bures Distance $D_B$', fontweight='bold')
axes[1].set_title('(B) Bures-Fidelity Relationship', fontweight='bold', loc='left')
axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.suptitle(f'3-Qubit: Pearson r={pearson_r:.4f}  Spearman ρ={spearman_r:.4f}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, 'fig1_geodesic_bures_3q.png'), dpi=200, bbox_inches='tight')
plt.close(); print("Saved fig1_geodesic_bures_3q.png")

# ── Figure 2: Training dynamics ───────────────────────────────────────────────
ep = history['epoch']
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].semilogy(ep, history['loss_recon'],  lw=2, color='#2E86AB', label='Recon')
axes[0].semilogy(ep, history['loss_metric'], lw=2, color='#A23B72', label='Metric')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss (log)')
axes[0].set_title('(A) Loss Components', fontweight='bold', loc='left')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(ep, history['val_fid'], lw=2, color='#F18F01')
axes[1].axhline(target_fidelity, color='red', ls='--', lw=1.5,
                label=f'Target {target_fidelity}')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Mean Fidelity')
axes[1].set_title('(B) Validation Fidelity', fontweight='bold', loc='left')
axes[1].set_ylim([0.4, 1.0]); axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].plot(ep, history['noise_mean'], lw=2, color='#6A4C93')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Mean noise param $\\bar{p}$')
axes[2].set_title('(C) Learned Depolarising Noise', fontweight='bold', loc='left')
axes[2].grid(True, alpha=0.3)
plt.suptitle('3-Qubit Training Dynamics', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, 'fig2_training_dynamics_3q.png'), dpi=200, bbox_inches='tight')
plt.close(); print("Saved fig2_training_dynamics_3q.png")

# ── Figure 3: Shot noise robustness ──────────────────────────────────────────
labels_s = list(shot_results.keys())
means_s  = [shot_results[k][0] for k in labels_s]
stds_s   = [shot_results[k][1] for k in labels_s]
x_pos    = np.arange(len(labels_s))
bar_cols = ['#2E86AB', '#F18F01', '#A23B72', '#6A4C93'][:len(labels_s)]

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x_pos, means_s, yerr=stds_s, capsize=5, color=bar_cols, alpha=0.85, edgecolor='black')
ax.axhline(means_s[0], color='red', ls='--', lw=1.5, label='Analytical baseline')
ax.set_xticks(x_pos); ax.set_xticklabels(labels_s, rotation=10, ha='right')
ax.set_ylabel('Mean Reconstruction Fidelity', fontweight='bold')
ax.set_title('Shot Noise Robustness — 3 Qubits', fontweight='bold')
ax.set_ylim([max(0.0, min(means_s) - 0.12), 1.0])
ax.legend(); ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, 'fig3_shot_noise_3q.png'), dpi=200, bbox_inches='tight')
plt.close(); print("Saved fig3_shot_noise_3q.png")

# ── Figure 4: Manifold analysis (4-panel) ────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
ev = pca_full.explained_variance_ratio_
n_show = min(15, len(ev))
axes[0,0].bar(range(1, n_show+1), ev[:n_show], alpha=0.75, color='#2E86AB', edgecolor='black')
ax2 = axes[0,0].twinx()
ax2.plot(range(1, n_show+1), np.cumsum(ev[:n_show]), 'ro-', lw=2, ms=5, label='Cumul.')
ax2.axhline(0.95, color='g', ls='--', alpha=0.7, label='95%')
axes[0,0].set_xlabel('PC'); axes[0,0].set_ylabel('Variance', color='#2E86AB')
ax2.set_ylabel('Cumulative', color='red')
axes[0,0].set_title('(A) PCA Spectrum', fontweight='bold', loc='left')
ax2.legend(loc='lower right')

axes[0,1].hist(curvatures, bins=40, color='#A23B72', alpha=0.75, edgecolor='black', density=True)
axes[0,1].axvline(np.mean(curvatures), color='red', ls='--', lw=2,
                  label=f'Mean={np.mean(curvatures):.4f}')
axes[0,1].set_xlabel('Local curvature κ'); axes[0,1].set_ylabel('Density')
axes[0,1].set_title('(B) Curvature Distribution', fontweight='bold', loc='left')
axes[0,1].legend()

sc2 = axes[1,0].scatter(pca2_v[:,0], pca2_v[:,1],
                        c=np.arange(n_vis), cmap='viridis', s=20, alpha=0.7)
axes[1,0].set_xlabel('PC 1'); axes[1,0].set_ylabel('PC 2')
axes[1,0].set_title('(C) Latent Space (PCA 2D)', fontweight='bold', loc='left')
plt.colorbar(sc2, ax=axes[1,0])

axes[1,1].scatter(bures_dists[:300], geo_dists[:300],
                  s=15, alpha=0.5, color='#F18F01', rasterized=True)
axes[1,1].plot(bures_line, a_fit*bures_line + b_fit, 'r-', lw=2,
               label=f'$R^2={r_squared:.3f}$')
axes[1,1].set_xlabel('Bures Distance'); axes[1,1].set_ylabel('Latent Distance')
axes[1,1].set_title('(D) Metric Correlation', fontweight='bold', loc='left')
axes[1,1].legend()

plt.suptitle('3-Qubit Manifold Geometry Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, 'fig4_manifold_analysis_3q.png'), dpi=200, bbox_inches='tight')
plt.close(); print("Saved fig4_manifold_analysis_3q.png")

# ── Figure 5: 3-D manifold comparison ────────────────────────────────────────
fig = plt.figure(figsize=(16, 7))

ax1 = fig.add_subplot(121, projection='3d')
ax1.scatter(q3d[:,0], q3d[:,1], q3d[:,2], c=colors, cmap='viridis', s=40, alpha=0.75)
nbrs_q = NearestNeighbors(n_neighbors=4).fit(q3d)
_, iq  = nbrs_q.kneighbors(q3d)
for i in range(n_vis):
    for j in iq[i][1:3]:
        ax1.plot([q3d[i,0],q3d[j,0]], [q3d[i,1],q3d[j,1]], [q3d[i,2],q3d[j,2]],
                 'gray', alpha=0.12, lw=0.5)
ax1.set_title('Quantum State Manifold\n(Bures MDS)', fontsize=13, fontweight='bold')
ax1.set_xlabel('Bures D1'); ax1.set_ylabel('Bures D2'); ax1.set_zlabel('Bures D3')
ax1.view_init(elev=20, azim=45)

ax2 = fig.add_subplot(122, projection='3d')
ax2.scatter(pca3_v[:,0], pca3_v[:,1], pca3_v[:,2], c=colors, cmap='viridis', s=40, alpha=0.75)
nbrs_l = NearestNeighbors(n_neighbors=4).fit(pca3_v)
_, il  = nbrs_l.kneighbors(pca3_v)
for i in range(n_vis):
    for j in il[i][1:3]:
        ax2.plot([pca3_v[i,0],pca3_v[j,0]],[pca3_v[i,1],pca3_v[j,1]],[pca3_v[i,2],pca3_v[j,2]],
                 'gray', alpha=0.12, lw=0.5)
ax2.set_title('Learned Latent Manifold\n(PCA 3D)', fontsize=13, fontweight='bold')
ax2.set_xlabel('PC 1'); ax2.set_ylabel('PC 2'); ax2.set_zlabel('PC 3')
ax2.view_init(elev=20, azim=45)

fig.text(0.5, 0.01,
         f'r = {pearson_r:.4f}  |  ρ = {spearman_r:.4f}  |  Procrustes RMSE = {rmse_proc:.4f}',
         ha='center', fontsize=12, style='italic',
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig(os.path.join(OUTDIR, 'fig5_manifold_3d_3q.png'), dpi=200, bbox_inches='tight')
plt.close(); print("Saved fig5_manifold_3d_3q.png")

# ── Figure 6: Fidelity distribution ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].hist(all_fids, bins=40, color='#2E86AB', alpha=0.75, edgecolor='black', density=True)
axes[0].axvline(val_mean_fid,   color='red',   ls='--', lw=2, label=f'Mean={val_mean_fid:.4f}')
axes[0].axvline(val_median_fid, color='green', ls='--', lw=2, label=f'Median={val_median_fid:.4f}')
axes[0].set_xlabel('Fidelity'); axes[0].set_ylabel('Density')
axes[0].set_title('(A) Fidelity Distribution', fontweight='bold', loc='left')
axes[0].legend()

sf    = np.sort(all_fids)
cdf   = np.arange(1, len(sf)+1) / len(sf)
axes[1].plot(sf, cdf, lw=2.5, color='#A23B72')
axes[1].axvline(0.90, color='red', ls='--', alpha=0.7, label='F=0.90')
axes[1].set_xlabel('Fidelity'); axes[1].set_ylabel('Cumulative Probability')
axes[1].set_title('(B) CDF', fontweight='bold', loc='left')
axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.suptitle('3-Qubit Reconstruction Fidelity', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, 'fig6_fidelity_dist_3q.png'), dpi=200, bbox_inches='tight')
plt.close(); print("Saved fig6_fidelity_dist_3q.png")

# ============================================================================
# Final Results Summary
# ============================================================================
print("\n" + "=" * 72)
print("RESULTS — 3-QUBIT GEOMETRIC LATENT SPACE TOMOGRAPHY")
print("=" * 72)
print(f"  Qubits      : {n_qubits}  |  Pauli observables: {n_pauli_nonid}")
print(f"  Decoder     : mixed-state (DepolarizingChannel, {n_noise_params} learned noise params)")
print(f"  Fidelity    : mean={val_mean_fid:.4f}  median={val_median_fid:.4f}")
print(f"  Pearson r   : {pearson_r:.4f}  |  Spearman ρ: {spearman_r:.4f}  |  R²: {r_squared:.4f}")
print(f"  Intrinsic d : {id_mean:.2f} ± {id_std:.2f} (MLE)  |  PCA-95%: {dim_95} dims")
print(f"  Curvature   : {np.mean(curvatures):.5f} ± {np.std(curvatures):.5f}")
print(f"  Procrustes  : RMSE = {rmse_proc:.4f}")
print(f"  Shot noise  :")
for lbl, (mf, sf) in shot_results.items():
    print(f"    {lbl:>25} : {mf:.4f} ± {sf:.4f}")
print("=" * 72)
print("All figures saved to", OUTDIR)


Generating training dataset...
Generating 2000 states with purity [0.85, 0.95]
Using 7 state types with EQUAL weights
  Generating 286 Depolarized states...
  Generating 286 Werner states...
  Generating 286 Isotropic states...
  Generating 286 AmpDamped states...
  Generating 286 PhaseDamped states...
  Generating 285 Thermal states...
  Generating 285 Separable states...

Dataset Statistics:
  Total samples: 2000
  Purity range: [0.840, 0.960], mean: 0.914

State type distribution:
    Depolarized    :  286 ( 14.3%)
    Werner         :  286 ( 14.3%)
    Isotropic      :  286 ( 14.3%)
    AmpDamped      :  286 ( 14.3%)
    PhaseDamped    :  286 ( 14.3%)
    Thermal        :  285 ( 14.2%)
    Separable      :  285 ( 14.2%)
Generating validation dataset...
Generating 500 states with purity [0.85, 0.95]
Using 7 state types with EQUAL weights
  Generating 72 Depolarized states...
  Generating 72 Werner states...
  Generating 72 Isotropic states...
  Generating 71 AmpDamped states...
  Ge